# 02 · Render geodesic domes in 3D

Shaded 3D renders of domes of increasing frequency, drawn with matplotlib's `Poly3DCollection`.
With `ipympl` installed you can drag every 3D plot to rotate it.

Script version: [`examples/02_plot_dome_3d.py`](../02_plot_dome_3d.py) · Needs: numpy, matplotlib, ipywidgets (optional: `ipympl` for live, rotatable figures)

<sub>SPDX-License-Identifier: AGPL-3.0-or-later · Copyright (C) 2022-2026 Masahiro Takatsuka. See the NOTICE file for attribution terms.</sub>

## Set-up

In [ ]:
import nb_setup  # makes mt.geodesicdome and examples/dome_utils.py importable from a checkout

WIDGET = nb_setup.setup()  # live figures with ipympl, static images otherwise

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from dome_utils import shade, unique_mesh
from ipywidgets import Checkbox, Dropdown, IntSlider, interact
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from nb_setup import LiveFigure

from mt.geodesicdome.grid.geodesicdome import GeodesicDome

## A shaded dome

Each face is coloured with simple Lambert shading from its outward normal (`dome_utils.shade`).

In [ ]:
BASE = np.array([0.27, 0.51, 0.71])


def draw_dome(ax, f, style='shaded', show_points=False, elev=20, azim=35):
    dome = GeodesicDome(f)
    points, faces, _ = unique_mesh(dome)
    tri = points[faces]                                   # (F, 3, 3) triangle corners
    normals = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
    normals /= np.linalg.norm(normals, axis=1, keepdims=True)

    if style == 'shaded':
        colours, edges = shade(normals)[:, None] * BASE, 'white'
    elif style == 'icosahedron face':
        ico = GeodesicDome(1)
        centres = ico.get_all_xyz()[ico.get_all_triangles().reshape(-1, 3)].mean(axis=1)
        parent = np.argmax(tri.mean(axis=1) @ centres.T, axis=1)
        colours = plt.cm.tab20(parent % 20)[:, :3] * shade(normals)[:, None]
        edges = 'white'
    else:                                                 # wireframe
        colours, edges = (0, 0, 0, 0), BASE

    ax.add_collection3d(Poly3DCollection(tri, facecolors=colours, edgecolors=edges, linewidths=0.4))
    if show_points:
        ax.scatter(*points.T, s=4, c='black', depthshade=True)
    ax.set_xlim(-1, 1), ax.set_ylim(-1, 1), ax.set_zlim(-1, 1)
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=elev, azim=azim)
    ax.set_axis_off()
    ax.set_title(f'frequency {f}\n{len(points)} vertices, {len(faces)} faces')

## Frequency 1, 2, 4 and 8

In [ ]:
FREQUENCIES = (1, 2, 4, 8)

fig = plt.figure(figsize=(14, 4.4))
for n, f in enumerate(FREQUENCIES, start=1):
    draw_dome(fig.add_subplot(1, len(FREQUENCIES), n, projection='3d'), f)
fig.suptitle('mtGeodesicDome: GeodesicDome (icosahedron-based)', fontsize=14)
fig.subplots_adjust(left=0, right=1, bottom=0, top=0.80, wspace=0)
plt.show()

### Try it

Change the frequency, the colouring and the camera. (With live figures you can also drag the dome.)

In [ ]:
live = LiveFigure(figsize=(6, 6))


@interact(f=IntSlider(value=4, min=1, max=20, description='frequency'),
          style=Dropdown(options=['shaded', 'icosahedron face', 'wireframe'], description='style'),
          show_points=Checkbox(value=False, description='show vertices'),
          elev=IntSlider(value=20, min=-90, max=90, step=5, description='elevation'),
          azim=IntSlider(value=35, min=-180, max=180, step=5, description='azimuth'))
def explore(f, style, show_points, elev, azim):
    fig = live.clear()
    draw_dome(fig.add_subplot(projection='3d'), f, style, show_points, elev, azim)
    fig.subplots_adjust(left=0, right=1, bottom=0, top=0.9)
    live.refresh()


live.show()